# Readwise to Workflowy Sync - Interaktivní Notebook

Tento notebook vám umožní testovat jednotlivé kroky synchronizace a diagnostikovat případné problémy.

## Obsah:
1. Instalace závislostí
2. Konfigurace API klíčů
3. Test Readwise připojení
4. Načtení highlights s tagem "todo"
5. Prozkoumání dat
6. Test Workflowy připojení
7. Vytvoření testovací položky
8. Hromadný import

## 1. Instalace závislostí

Spusťte tuto buňku pro instalaci potřebných knihoven.

In [ ]:
# Instalace závislostí
!pip install requests python-dotenv wfapi

## 2. Konfigurace API klíčů

Zadejte své API klíče přímo zde, nebo je načtěte ze souboru `.env`.

In [ ]:
import os
from dotenv import load_dotenv

# Načtení z .env souboru (pokud existuje)
load_dotenv()

# ============================================
# MOŽNOST 1: Načíst z .env souboru (doporučeno)
# ============================================
READWISE_API_TOKEN = os.getenv("READWISE_API_TOKEN")

# Pro wfapi verzi:
WORKFLOWY_USERNAME = os.getenv("WORKFLOWY_USERNAME")
WORKFLOWY_PASSWORD = os.getenv("WORKFLOWY_PASSWORD")

# Pro přímé API:
WORKFLOWY_BEARER_TOKEN = os.getenv("WORKFLOWY_BEARER_TOKEN")
WORKFLOWY_SESSION_ID = os.getenv("WORKFLOWY_SESSION_ID")

# ============================================
# MOŽNOST 2: Zadat přímo zde (odkomentujte a vyplňte)
# ============================================
# READWISE_API_TOKEN = "váš_readwise_token"
# WORKFLOWY_USERNAME = "váš_email@example.com"
# WORKFLOWY_PASSWORD = "vaše_heslo"
# WORKFLOWY_BEARER_TOKEN = "váš_bearer_token"
# WORKFLOWY_SESSION_ID = "váš_session_id"

# Kontrola konfigurace
print("Konfigurace:")
print(f"  READWISE_API_TOKEN: {'✓ nastaveno' if READWISE_API_TOKEN else '✗ chybí'}")
print(f"  WORKFLOWY_USERNAME: {'✓ nastaveno' if WORKFLOWY_USERNAME else '✗ chybí'}")
print(f"  WORKFLOWY_PASSWORD: {'✓ nastaveno' if WORKFLOWY_PASSWORD else '✗ chybí'}")
print(f"  WORKFLOWY_BEARER_TOKEN: {'✓ nastaveno' if WORKFLOWY_BEARER_TOKEN else '✗ chybí'}")
print(f"  WORKFLOWY_SESSION_ID: {'✓ nastaveno' if WORKFLOWY_SESSION_ID else '✗ chybí'}")

## 3. Test Readwise připojení

Ověříme, že váš Readwise API token funguje.

In [ ]:
import requests

def test_readwise_connection(token):
    """Test Readwise API connection."""
    if not token:
        print("❌ READWISE_API_TOKEN není nastaven!")
        return False
    
    url = "https://readwise.io/api/v2/auth/"
    headers = {"Authorization": f"Token {token}"}
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 204:
            print("✅ Readwise připojení úspěšné!")
            return True
        else:
            print(f"❌ Readwise chyba: Status {response.status_code}")
            print(f"   Response: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Chyba připojení: {e}")
        return False

# Test
readwise_ok = test_readwise_connection(READWISE_API_TOKEN)

## 4. Načtení highlights s tagem "todo"

Načteme všechny vaše highlights, které mají tag "todo".

In [ ]:
def get_highlights_with_tag(token, tag="todo"):
    """Fetch all highlights with a specific tag."""
    highlights = []
    url = "https://readwise.io/api/v2/highlights/"
    headers = {"Authorization": f"Token {token}"}
    
    print(f"Načítám highlights s tagem '{tag}'...")
    page = 1
    
    while url:
        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            data = response.json()
            
            # Filtrování podle tagu
            for highlight in data.get("results", []):
                tags = highlight.get("tags", [])
                tag_names = [t.get("name", "").lower() for t in tags]
                
                if tag.lower() in tag_names:
                    highlights.append(highlight)
            
            print(f"  Stránka {page}: Celkem nalezeno {len(highlights)} highlights s tagem '{tag}'")
            
            # Další stránka
            url = data.get("next")
            page += 1
            
        except Exception as e:
            print(f"❌ Chyba: {e}")
            break
    
    print(f"\n✅ Celkem nalezeno: {len(highlights)} highlights s tagem '{tag}'")
    return highlights

# Načtení highlights
highlights = get_highlights_with_tag(READWISE_API_TOKEN, "todo")

## 5. Prozkoumání dat

Podívejme se, jak vypadají načtené highlights.

In [ ]:
import json

# Počet highlights
print(f"Počet highlights: {len(highlights)}")
print()

# Zobrazit první 3 highlights
if highlights:
    print("=" * 70)
    print("UKÁZKA PRVNÍCH 3 HIGHLIGHTS:")
    print("=" * 70)
    
    for i, h in enumerate(highlights[:3], 1):
        print(f"\n--- Highlight {i} ---")
        print(f"Text: {h.get('text', 'N/A')[:200]}..." if len(h.get('text', '')) > 200 else f"Text: {h.get('text', 'N/A')}")
        print(f"Kniha/Zdroj: {h.get('book_title', 'N/A')}")
        print(f"Autor: {h.get('author', 'N/A')}")
        print(f"URL: {h.get('url', 'N/A')}")
        print(f"Poznámka: {h.get('note', 'N/A')}")
        print(f"Tagy: {[t.get('name') for t in h.get('tags', [])]}")
else:
    print("⚠️ Žádné highlights nebyly nalezeny!")

In [ ]:
# Zobrazit kompletní strukturu jednoho highlightu (pro debugging)
if highlights:
    print("KOMPLETNÍ STRUKTURA PRVNÍHO HIGHLIGHTU:")
    print(json.dumps(highlights[0], indent=2, ensure_ascii=False))

## 6. Test Workflowy připojení

### Varianta A: Použití wfapi knihovny (DOPORUČENO)

In [ ]:
# Test wfapi připojení
wf = None

try:
    from wfapi import Workflowy
    
    if WORKFLOWY_USERNAME and WORKFLOWY_PASSWORD:
        print("Připojuji se k Workflowy pomocí wfapi...")
        wf = Workflowy(username=WORKFLOWY_USERNAME, password=WORKFLOWY_PASSWORD)
        print("✅ Workflowy (wfapi) připojení úspěšné!")
        print(f"   Root node má {len(list(wf.root))} přímých potomků")
    else:
        print("⚠️ WORKFLOWY_USERNAME nebo WORKFLOWY_PASSWORD není nastaveno")
        print("   Přeskakuji wfapi test...")
        
except ImportError:
    print("⚠️ wfapi knihovna není nainstalována")
    print("   Spusťte: pip install wfapi")
except Exception as e:
    print(f"❌ Chyba připojení k Workflowy (wfapi): {e}")

### Varianta B: Použití přímého API

In [ ]:
# Test přímého Workflowy API
workflowy_session = None

def test_workflowy_direct_api():
    """Test direct Workflowy API connection."""
    session = requests.Session()
    
    if WORKFLOWY_BEARER_TOKEN:
        session.headers.update({
            "Authorization": f"Bearer {WORKFLOWY_BEARER_TOKEN}",
            "Content-Type": "application/json"
        })
        auth_method = "Bearer Token"
    elif WORKFLOWY_SESSION_ID:
        session.cookies.set("sessionid", WORKFLOWY_SESSION_ID)
        session.headers.update({"Content-Type": "application/json"})
        auth_method = "Session ID"
    else:
        print("⚠️ Žádná autentizace pro přímé API není nastavena")
        return None
    
    print(f"Testuji Workflowy přímé API ({auth_method})...")
    
    # Zkusíme různé endpointy
    test_urls = [
        "https://workflowy.com/api/create",
        "https://workflowy.com/api/nodes",
        "https://beta.workflowy.com/api/create",
    ]
    
    for url in test_urls:
        try:
            # Test s prázdným payloadem
            response = session.post(url, json={"name": "test"})
            print(f"  {url}")
            print(f"    Status: {response.status_code}")
            print(f"    Response: {response.text[:200]}..." if len(response.text) > 200 else f"    Response: {response.text}")
            print()
        except Exception as e:
            print(f"  {url}")
            print(f"    ❌ Chyba: {e}")
            print()
    
    return session

workflowy_session = test_workflowy_direct_api()

## 7. Vytvoření testovací položky

Zkusíme vytvořit jednu testovací položku ve Workflowy.

### Varianta A: Pomocí wfapi

In [ ]:
# Vytvoření testovací položky pomocí wfapi
if wf:
    try:
        print("Vytvářím testovací položku pomocí wfapi...")
        
        # Vytvoření nové položky
        test_node = wf.create(wf.root)
        wf.edit(test_node, "[TEST] Readwise sync test - můžete smazat")
        wf.edit(test_node, description="Toto je testovací položka vytvořená z Jupyter notebooku")
        
        print("✅ Testovací položka úspěšně vytvořena!")
        print("   Zkontrolujte svůj Workflowy a smažte ji.")
        
    except Exception as e:
        print(f"❌ Chyba při vytváření položky: {e}")
else:
    print("⚠️ wfapi není připojeno. Zkuste nejdřív buňku 6A.")

### Varianta B: Pomocí přímého API

In [ ]:
# Vytvoření testovací položky pomocí přímého API
if workflowy_session:
    print("Vytvářím testovací položku pomocí přímého API...")
    
    payload = {
        "name": "[TEST] Readwise sync test - můžete smazat",
        "description": "Testovací položka z Jupyter notebooku"
    }
    
    try:
        response = workflowy_session.post(
            "https://workflowy.com/api/create",
            json=payload
        )
        
        print(f"Status: {response.status_code}")
        print(f"Response: {response.text}")
        
        if response.status_code in [200, 201]:
            print("\n✅ Testovací položka úspěšně vytvořena!")
        else:
            print("\n❌ Vytvoření selhalo. Zkuste použít wfapi verzi (buňka 7A).")
            
    except Exception as e:
        print(f"❌ Chyba: {e}")
else:
    print("⚠️ Přímé API session není k dispozici.")

## 8. Hromadný import highlights

Pokud testy v sekci 7 proběhly úspěšně, můžete spustit hromadný import.

### Varianta A: Pomocí wfapi (DOPORUČENO)

In [ ]:
def import_highlights_wfapi(wf_client, highlights_list, limit=None):
    """
    Import highlights do Workflowy pomocí wfapi.
    
    Args:
        wf_client: Workflowy client instance
        highlights_list: List of highlights from Readwise
        limit: Optional limit (pro testování)
    """
    if limit:
        highlights_list = highlights_list[:limit]
        print(f"⚠️ Omezeno na prvních {limit} highlights (pro testování)")
    
    created = 0
    failed = 0
    errors = []
    
    print(f"\nImportuji {len(highlights_list)} highlights do Workflowy...\n")
    
    for idx, highlight in enumerate(highlights_list, 1):
        try:
            text = highlight.get("text", "")
            book_title = highlight.get("book_title", "")
            author = highlight.get("author", "")
            url = highlight.get("url", "")
            note = highlight.get("note", "")
            
            # Sestavení popisu
            desc_parts = []
            if book_title:
                desc_parts.append(f"📚 {book_title}")
            if author:
                desc_parts.append(f"✍️ {author}")
            if url:
                desc_parts.append(f"🔗 {url}")
            if note:
                desc_parts.append(f"📝 {note}")
            
            description = " | ".join(desc_parts) if desc_parts else None
            
            # Vytvoření položky
            node = wf_client.create(wf_client.root)
            wf_client.edit(node, text)
            if description:
                wf_client.edit(node, description=description)
            
            created += 1
            print(f"  [{idx}/{len(highlights_list)}] ✅ {text[:50]}...")
            
        except Exception as e:
            failed += 1
            errors.append({"idx": idx, "text": text[:50], "error": str(e)})
            print(f"  [{idx}/{len(highlights_list)}] ❌ {text[:50]}... - {str(e)[:50]}")
    
    print(f"\n{'='*60}")
    print(f"HOTOVO!")
    print(f"  ✅ Úspěšně vytvořeno: {created}")
    print(f"  ❌ Selhalo: {failed}")
    print(f"{'='*60}")
    
    if errors:
        print(f"\nPrvní chyby:")
        for err in errors[:5]:
            print(f"  - [{err['idx']}] {err['text']}: {err['error']}")
    
    return created, failed

In [ ]:
# TESTOVACÍ IMPORT - pouze první 3 highlights
# Odkomentujte a spusťte pro test

if wf and highlights:
    print("🧪 TESTOVACÍ IMPORT (pouze 3 položky)")
    import_highlights_wfapi(wf, highlights, limit=3)
else:
    print("⚠️ wfapi není připojeno nebo nejsou žádné highlights")

In [ ]:
# PLNÝ IMPORT - všechny highlights
# ⚠️ POZOR: Toto vytvoří všech {len(highlights)} položek!
# Odkomentujte následující řádky pro spuštění plného importu

# if wf and highlights:
#     print(f"🚀 PLNÝ IMPORT ({len(highlights)} položek)")
#     confirm = input("Opravdu chcete importovat všechny highlights? (ano/ne): ")
#     if confirm.lower() == "ano":
#         import_highlights_wfapi(wf, highlights)
#     else:
#         print("Import zrušen.")
# else:
#     print("⚠️ wfapi není připojeno nebo nejsou žádné highlights")

print("⚠️ Plný import je zakomentovaný. Odkomentujte kód výše pro spuštění.")

## 9. Debugging pomocníci

Další užitečné funkce pro diagnostiku problémů.

In [ ]:
# Zobrazit všechny tagy z Readwise
def list_all_tags(token):
    """List all tags from Readwise."""
    url = "https://readwise.io/api/v2/highlights/"
    headers = {"Authorization": f"Token {token}"}
    
    all_tags = set()
    
    while url:
        response = requests.get(url, headers=headers)
        data = response.json()
        
        for highlight in data.get("results", []):
            for tag in highlight.get("tags", []):
                all_tags.add(tag.get("name", ""))
        
        url = data.get("next")
    
    return sorted(all_tags)

print("Načítám všechny tagy z Readwise...")
tags = list_all_tags(READWISE_API_TOKEN)
print(f"\nNalezené tagy ({len(tags)}):")
for tag in tags:
    print(f"  - {tag}")

In [ ]:
# Export highlights do JSON souboru (pro zálohu)
import json

if highlights:
    filename = "readwise_todo_highlights.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(highlights, f, ensure_ascii=False, indent=2)
    print(f"✅ Highlights exportovány do '{filename}'")
    print(f"   Počet: {len(highlights)}")
else:
    print("⚠️ Žádné highlights k exportu")

---

## Řešení problémů

### Readwise
- **401 Unauthorized**: Zkontrolujte, zda máte správný API token z https://readwise.io/access_token
- **Žádné highlights**: Ujistěte se, že máte highlights s tagem "todo" (tag je case-insensitive)

### Workflowy (wfapi)
- **Login failed**: Zkontrolujte username a heslo
- **2FA**: wfapi nepodporuje dvoufaktorovou autentizaci, vypněte ji v nastavení Workflowy

### Workflowy (přímé API)
- **403/401**: Bearer token nebo Session ID je neplatný nebo vypršel
- **Session ID**: Vyprší po odhlášení z Workflowy, obnovte ho

### Doporučení
**Použijte wfapi verzi** - je spolehlivější a lépe testovaná než přímé API volání.